# End-to-End OMOP CDM Transformation

This notebook demonstrates the complete pipeline for transforming raw hospital data
into the OMOP Common Data Model (CDM) v5.4 using the Portiere SDK.

The pipeline consists of five stages:

1. **Ingest** -- Load source data files into the project
2. **Profile** -- Analyze source data structure and content
3. **Schema Map** -- Map source columns to OMOP CDM table columns
4. **Concept Map** -- Translate source codes to standard OMOP concepts
5. **ETL + Validate** -- Generate and execute the transformation, then validate output

## Overview

We will transform a typical hospital dataset consisting of four source files:

| Source File | Description | Target OMOP Table |
|---|---|---|
| `patients.csv` | Patient demographics | `person` |
| `encounters.csv` | Hospital visits/admissions | `visit_occurrence` |
| `diagnoses.csv` | Diagnosis codes (ICD-10) | `condition_occurrence` |
| `medications.csv` | Prescribed medications | `drug_exposure` |

The OMOP CDM v5.4 standardizes clinical data into a common schema, enabling
cross-institutional research using tools from the OHDSI community (ATLAS,
ACHILLES, CohortDiagnostics, etc.).

## Stage 1: Project Setup and Configuration

We begin by configuring the Portiere project with custom thresholds for schema
and concept mapping. These thresholds control the confidence levels at which
mappings are auto-accepted versus flagged for review.

In [1]:
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("portiere").getOrCreate()
    print(f"SparkSession created: {spark.version}")
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    spark = None
    print("PySpark not installed — Spark cells will be skipped. Install with: pip install pyspark")


PySpark not installed — Spark cells will be skipped. Install with: pip install pyspark


import portiere
from portiere import PortiereConfig, KnowledgeLayerConfig, ThresholdsConfig
from portiere.config import ConceptMappingThresholds, SchemaMappingThresholds
from portiere.engines import SparkEngine

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
    thresholds=ThresholdsConfig(
        schema_mapping=SchemaMappingThresholds(auto_accept=0.85),
        concept_mapping=ConceptMappingThresholds(auto_accept=0.90),
    ),
)

project = portiere.init(name="Hospital OMOP ETL", engine=SparkEngine(spark), config=config)
print(f"Project: {project.name}")
print(f"Schema auto-accept threshold: {config.thresholds.schema_mapping.auto_accept}")
print(f"Concept auto-accept threshold: {config.thresholds.concept_mapping.auto_accept}")

In [2]:
from pathlib import Path
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


In [3]:
import portiere
from portiere import PortiereConfig, KnowledgeLayerConfig, ThresholdsConfig
from portiere.config import ConceptMappingThresholds, SchemaMappingThresholds

if SPARK_AVAILABLE:
    from portiere.engines import SparkEngine
    engine = SparkEngine(spark)
else:
    from portiere.engines import PolarsEngine
    engine = PolarsEngine()
    print("PySpark not installed — using PolarsEngine instead")

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
    thresholds=ThresholdsConfig(
        schema_mapping=SchemaMappingThresholds(auto_accept=0.90, needs_review=0.70),
        concept_mapping=ConceptMappingThresholds(auto_accept=0.92, needs_review=0.75),
    ),
)

project = portiere.init(
    name="OMOP End-to-End Demo",
    engine=engine,
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config,
)

print(f"Project: {project.name}")
print(f"Engine: {type(engine).__name__}")


2026-04-16 00:27:45 [info     ] PolarsEngine initialized      


PySpark not installed — using PolarsEngine instead
2026-04-16 00:27:45 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project: OMOP End-to-End Demo
Engine: PolarsEngine


## Stage 2: Ingest Source Data

Add all source files to the project. Each source is assigned a name for easy
reference throughout the pipeline. Portiere will automatically profile the
data to understand column types, value distributions, and data quality.

In [4]:
patients = project.add_source("example_assets/08_end_to_end_omop/patients.csv", name="patients")
encounters = project.add_source("example_assets/08_end_to_end_omop/encounters.csv", name="encounters")
diagnoses = project.add_source("example_assets/08_end_to_end_omop/diagnoses.csv", name="diagnoses")
medications = project.add_source("example_assets/08_end_to_end_omop/medications.csv", name="medications")

sources = [patients, encounters, diagnoses, medications]
for src in sources:
    print(f"Loaded: {src['name']}")

2026-04-16 00:27:45 [info     ] project.source_added           project='OMOP End-to-End Demo' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:27:48 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


2026-04-16 00:27:48 [info     ] project.profiled               source=patients


2026-04-16 00:27:48 [info     ] project.source_added           project='OMOP End-to-End Demo' source=encounters


...

2026-04-16 00:27:48 [info     ] project.source_added           project='OMOP End-to-End Demo' source=medications


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:27:48 [info     ] gx_profiler.profiled           columns=7 rows=15 source=medications


2026-04-16 00:27:48 [info     ] project.profiled               source=medications


Loaded: patients
Loaded: encounters
Loaded: diagnoses
Loaded: medications


## Stage 3: Schema Mapping

Schema mapping determines how source columns map to OMOP CDM table columns.
Portiere uses the column names, data types, and sample values to suggest
the best target table and column for each source field.

### 3a. Map Patient Demographics to `person`

In [5]:
# Schema mapping for patients -> person table
patients_schema = project.map_schema(patients)

print("Patients Schema Mapping:")
print("=" * 70)
for item in patients_schema.items:
    status_marker = "[AUTO]" if item.confidence >= 0.85 else "[REVIEW]"
    print(f"  {status_marker} {item.source_column:25s} -> "
          f"{item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:27:48 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:27:48 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:27:48 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:27:48 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:27:50 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:27:50 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:27:50 [info     ] local_storage.schema_mapping_saved items_count=11 project='OMOP End-to-End Demo'


2026-04-16 00:27:50 [info     ] project.schema_mapped          auto=10 project='OMOP End-to-End Demo' total=11


Patients Schema Mapping:
  [AUTO] patient_id                -> person.person_id (confidence=0.95)
  [AUTO] first_name                -> person.person_source_value (confidence=0.95)
  [REVIEW] last_name                 -> person.person_source_value (confidence=0.50)
  [AUTO] date_of_birth             -> person.birth_datetime (confidence=0.95)
  [AUTO] gender                    -> person.gender_concept_id (confidence=0.95)
  [AUTO] race                      -> person.race_concept_id (confidence=0.95)
  [AUTO] ethnicity                 -> person.ethnicity_concept_id (confidence=0.95)
  [AUTO] address                   -> location.address_1 (confidence=0.95)
  [AUTO] city                      -> location.city (confidence=0.95)
  [AUTO] state                     -> location.state (confidence=0.95)
  [AUTO] zip_code                  -> location.zip (confidence=0.95)


### 3b. Map Encounters to `visit_occurrence`

In [6]:
# Schema mapping for encounters -> visit_occurrence
encounters_schema = project.map_schema(encounters)

print("Encounters Schema Mapping:")
print("=" * 70)
for item in encounters_schema.items:
    status_marker = "[AUTO]" if item.confidence >= 0.85 else "[REVIEW]"
    print(f"  {status_marker} {item.source_column:25s} -> "
          f"{item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:27:50 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:27:50 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:27:50 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:27:50 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:27:50 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:27:50 [info     ] Stage 2 complete               auto=7 review=0 total=8


2026-04-16 00:27:50 [info     ] local_storage.schema_mapping_saved items_count=8 project='OMOP End-to-End Demo'


2026-04-16 00:27:50 [info     ] project.schema_mapped          auto=7 project='OMOP End-to-End Demo' total=8


Encounters Schema Mapping:
  [AUTO] encounter_id              -> visit_occurrence.visit_occurrence_id (confidence=0.95)
  [AUTO] patient_id                -> person.person_id (confidence=0.95)
  [AUTO] encounter_type            -> visit_occurrence.visit_concept_id (confidence=0.95)
  [AUTO] admission_date            -> visit_occurrence.visit_start_date (confidence=0.95)
  [AUTO] discharge_date            -> visit_occurrence.visit_end_date (confidence=0.95)
  [AUTO] facility                  -> visit_occurrence.care_site_id (confidence=0.95)
  [REVIEW] department                -> visit_occurrence.care_site_id (confidence=0.50)
  [AUTO] attending_provider        -> visit_occurrence.provider_id (confidence=0.95)


### 3c. Map Diagnoses to `condition_occurrence`

In [7]:
# Schema mapping for diagnoses -> condition_occurrence
diagnoses_schema = project.map_schema(diagnoses)

print("Diagnoses Schema Mapping:")
print("=" * 70)
for item in diagnoses_schema.items:
    status_marker = "[AUTO]" if item.confidence >= 0.85 else "[REVIEW]"
    print(f"  {status_marker} {item.source_column:25s} -> "
          f"{item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:27:50 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:27:50 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:27:50 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:27:50 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:27:50 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:27:50 [info     ] Stage 2 complete               auto=5 review=0 total=6


2026-04-16 00:27:50 [info     ] local_storage.schema_mapping_saved items_count=6 project='OMOP End-to-End Demo'


2026-04-16 00:27:50 [info     ] project.schema_mapped          auto=5 project='OMOP End-to-End Demo' total=6


Diagnoses Schema Mapping:
  [AUTO] encounter_id              -> visit_occurrence.visit_occurrence_id (confidence=0.95)
  [AUTO] patient_id                -> person.person_id (confidence=0.95)
  [AUTO] diagnosis_code            -> condition_occurrence.condition_source_value (confidence=0.95)
  [REVIEW] diagnosis_description     -> condition_occurrence.condition_source_value (confidence=0.50)
  [AUTO] diagnosis_type            -> condition_occurrence.condition_type_concept_id (confidence=0.95)
  [AUTO] diagnosis_date            -> condition_occurrence.condition_start_date (confidence=0.95)


### 3d. Map Medications to `drug_exposure`

In [8]:
# Schema mapping for medications -> drug_exposure
medications_schema = project.map_schema(medications)

print("Medications Schema Mapping:")
print("=" * 70)
for item in medications_schema.items:
    status_marker = "[AUTO]" if item.confidence >= 0.85 else "[REVIEW]"
    print(f"  {status_marker} {item.source_column:25s} -> "
          f"{item.target_table}.{item.target_column} "
          f"(confidence={item.confidence:.2f})")

2026-04-16 00:27:50 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:27:50 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:27:50 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:27:50 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:27:50 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:27:50 [info     ] Stage 2 complete               auto=6 review=0 total=7


2026-04-16 00:27:50 [info     ] local_storage.schema_mapping_saved items_count=7 project='OMOP End-to-End Demo'


2026-04-16 00:27:50 [info     ] project.schema_mapped          auto=6 project='OMOP End-to-End Demo' total=7


Medications Schema Mapping:
  [AUTO] prescription_id           -> drug_exposure.drug_exposure_id (confidence=0.95)
  [AUTO] patient_id                -> person.person_id (confidence=0.95)
  [AUTO] drug_code                 -> drug_exposure.drug_source_value (confidence=0.95)
  [REVIEW] drug_name                 -> drug_exposure.drug_source_value (confidence=0.50)
  [AUTO] quantity                  -> drug_exposure.quantity (confidence=0.95)
  [AUTO] prescription_date         -> drug_exposure.drug_exposure_start_date (confidence=0.95)
  [AUTO] prescriber_id             -> drug_exposure.provider_id (confidence=0.95)


### 3e. Review Schema Mapping Results

Inspect items that need review and approve or correct them before proceeding.

In [9]:
# Collect all schema mappings for review
all_schema_maps = {
    "patients": patients_schema,
    "encounters": encounters_schema,
    "diagnoses": diagnoses_schema,
    "medications": medications_schema,
}

print("Schema Mapping Summary")
print("=" * 50)
for name, schema_map in all_schema_maps.items():
    total = len(schema_map.items)
    auto = sum(1 for item in schema_map.items if item.confidence >= 0.85)
    review = total - auto
    print(f"  {name:15s}: {auto}/{total} auto-mapped, {review} need review")

print("\nItems needing review:")
for name, schema_map in all_schema_maps.items():
    for item in schema_map.items:
        if item.confidence < 0.85:
            print(f"  [{name}] {item.source_column} -> "
                  f"{item.target_table}.{item.target_column} "
                  f"(confidence={item.confidence:.2f})")

Schema Mapping Summary
  patients       : 10/11 auto-mapped, 1 need review
  encounters     : 7/8 auto-mapped, 1 need review
  diagnoses      : 5/6 auto-mapped, 1 need review
  medications    : 6/7 auto-mapped, 1 need review

Items needing review:
  [patients] last_name -> person.person_source_value (confidence=0.50)
  [encounters] department -> visit_occurrence.care_site_id (confidence=0.50)
  [diagnoses] diagnosis_description -> condition_occurrence.condition_source_value (confidence=0.50)
  [medications] drug_name -> drug_exposure.drug_source_value (confidence=0.50)


## Stage 4: Concept Mapping

With schema mappings in place, we now map source codes to standard OMOP concepts.
This stage handles the vocabulary translation that is central to OMOP CDM compliance.

### 4a. Map Diagnosis Codes (ICD-10 to SNOMED)

In [10]:
# Map diagnosis codes from the diagnoses source file
diagnosis_codes = [
    {"code": "E11.9", "description": "Type 2 diabetes mellitus, without complications", "count": 1523},
    {"code": "I10", "description": "Essential (primary) hypertension", "count": 2891},
    {"code": "J18.9", "description": "Pneumonia, unspecified organism", "count": 456},
    {"code": "N18.6", "description": "End stage renal disease", "count": 234},
    {"code": "K21.0", "description": "Gastro-esophageal reflux disease with esophagitis", "count": 678},
    {"code": "M54.5", "description": "Low back pain", "count": 1345},
    {"code": "F32.9", "description": "Major depressive disorder, single episode, unspecified", "count": 891},
    {"code": "J44.1", "description": "Chronic obstructive pulmonary disease with acute exacerbation", "count": 312},
]

dx_concept_map = project.map_concepts(
    codes=diagnosis_codes,
    vocabularies=["SNOMED", "ICD10CM"],
)

dx_summary = dx_concept_map.summary()
print("Diagnosis Concept Mapping")
print("=" * 50)
print(f"  Total:          {dx_summary['total']}")
print(f"  Auto-mapped:    {dx_summary['auto_mapped']}")
print(f"  Needs review:   {dx_summary['needs_review']}")
print(f"  Auto rate:      {dx_summary['auto_rate']:.1%}")
print(f"  Coverage:       {dx_summary['coverage']:.1%}")

2026-04-16 00:27:50 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=bm25s


2026-04-16 00:27:50 [info     ] knowledge.creating_backend     backend=bm25s path=example_assets/vocabulary_download_v5/indexes/concepts.json


Split strings:   0%|          | 0/623910 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-16 00:28:00 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=8


2026-04-16 00:28:00 [info     ] local_storage.concept_mapping_saved items_count=8 project='OMOP End-to-End Demo'


2026-04-16 00:28:00 [info     ] project.concepts_mapped        auto_rate=100.0 project='OMOP End-to-End Demo' total=8


Diagnosis Concept Mapping
  Total:          8
  Auto-mapped:    8
  Needs review:   0
  Auto rate:      10000.0%
  Coverage:       10000.0%


### 4b. Map Medication Codes (Local to RxNorm)

In [11]:
# Map medication codes to RxNorm
medication_codes = [
    {"code": "MET500", "description": "Metformin 500mg oral tablet", "count": 2300},
    {"code": "LISIN10", "description": "Lisinopril 10mg oral tablet", "count": 1850},
    {"code": "ATOR20", "description": "Atorvastatin 20mg oral tablet", "count": 1600},
    {"code": "AMOX500", "description": "Amoxicillin 500mg oral capsule", "count": 900},
    {"code": "OMEP20", "description": "Omeprazole 20mg oral capsule", "count": 1100},
    {"code": "ASA81", "description": "Aspirin 81mg oral tablet", "count": 3200},
    {"code": "HYDRO25", "description": "Hydrochlorothiazide 25mg oral tablet", "count": 950},
]

rx_concept_map = project.map_concepts(
    codes=medication_codes,
    vocabularies=["RxNorm"],
)

rx_summary = rx_concept_map.summary()
print("Medication Concept Mapping")
print("=" * 50)
print(f"  Total:          {rx_summary['total']}")
print(f"  Auto-mapped:    {rx_summary['auto_mapped']}")
print(f"  Needs review:   {rx_summary['needs_review']}")
print(f"  Auto rate:      {rx_summary['auto_rate']:.1%}")
print(f"  Coverage:       {rx_summary['coverage']:.1%}")

2026-04-16 00:28:00 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=bm25s


2026-04-16 00:28:00 [info     ] knowledge.creating_backend     backend=bm25s path=example_assets/vocabulary_download_v5/indexes/concepts.json


Split strings:   0%|          | 0/623910 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-16 00:28:09 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=7


2026-04-16 00:28:09 [info     ] local_storage.concept_mapping_saved items_count=7 project='OMOP End-to-End Demo'


2026-04-16 00:28:09 [info     ] project.concepts_mapped        auto_rate=100.0 project='OMOP End-to-End Demo' total=7


Medication Concept Mapping
  Total:          7
  Auto-mapped:    7
  Needs review:   0
  Auto rate:      10000.0%
  Coverage:       10000.0%


### 4c. Review and Finalize Concept Mappings

Review items flagged for human attention, then finalize all mappings.

In [12]:
# Review diagnosis concept mappings
print("Diagnosis Mappings -- Review Items:")
print("-" * 80)
for item in dx_concept_map.needs_review():
    print(f"  {item.source_code}: {item.source_description}")
    print(f"    -> {item.target_concept_name} (confidence={item.confidence:.2f})")
    if item.candidates:
        for i, c in enumerate(item.candidates[:3]):
            marker = " <<" if i == 0 else ""
            print(f"    [{i}] {c.concept_name} ({c.vocabulary_id}, score={c.score:.3f}){marker}")
    print()

# Review medication concept mappings
print("Medication Mappings -- Review Items:")
print("-" * 80)
for item in rx_concept_map.needs_review():
    print(f"  {item.source_code}: {item.source_description}")
    print(f"    -> {item.target_concept_name} (confidence={item.confidence:.2f})")
    if item.candidates:
        for i, c in enumerate(item.candidates[:3]):
            marker = " <<" if i == 0 else ""
            print(f"    [{i}] {c.concept_name} ({c.vocabulary_id}, score={c.score:.3f}){marker}")
    print()

Diagnosis Mappings -- Review Items:
--------------------------------------------------------------------------------
Medication Mappings -- Review Items:
--------------------------------------------------------------------------------


In [13]:
# Approve all review items with their top candidate
dx_concept_map.approve_all()
rx_concept_map.approve_all()

# Finalize both concept mappings
dx_concept_map.finalize()
rx_concept_map.finalize()

print("Concept mappings finalized.")
print(f"  Diagnoses:   {dx_concept_map.summary()['auto_mapped']}/{dx_concept_map.summary()['total']} mapped")
print(f"  Medications: {rx_concept_map.summary()['auto_mapped']}/{rx_concept_map.summary()['total']} mapped")

Concept mappings finalized.
  Diagnoses:   8/8 mapped
  Medications: 7/7 mapped


## Stage 5: Run ETL Transformation

With schema and concept mappings finalized, we can now execute the ETL
(Extract, Transform, Load) process. Portiere generates the transformation
logic and produces OMOP-compliant output files.

The ETL process will:
1. Read each source file
2. Apply schema mappings (rename/transform columns)
3. Apply concept mappings (translate codes to standard concepts)
4. Generate OMOP CDM output tables
5. Write output files to the specified directory

In [14]:
# Combine schema mappings for the ETL
# In a real pipeline, you would pass the appropriate schema and concept
# mappings for each source. Here we demonstrate with the patients source.
schema_map = patients_schema
concept_map = dx_concept_map

etl_result = project.run_etl(
    patients,
    output_dir="./omop_output",
    schema_mapping=schema_map,
    concept_mapping=concept_map,
)

print("ETL completed.")
print(f"  Output directory: ./omop_output")
print(f"  Result: {etl_result}")

2026-04-16 00:28:09 [info     ] Reading source                 format=csv path=example_assets/08_end_to_end_omop/patients.csv


2026-04-16 00:28:09 [info     ] Processing table               columns=6 table=person


2026-04-16 00:28:09 [info     ] Processing table               columns=4 table=location


2026-04-16 00:28:09 [info     ] ETL complete                   duration=0.004082 success=True


2026-04-16 00:28:09 [info     ] project.etl_complete           output_dir=./omop_output project='OMOP End-to-End Demo'


ETL completed.
  Output directory: ./omop_output
  Result: success=True started_at=datetime.datetime(2026, 4, 15, 17, 28, 9, 385354, tzinfo=datetime.timezone.utc) completed_at=datetime.datetime(2026, 4, 15, 17, 28, 9, 389436, tzinfo=datetime.timezone.utc) duration_seconds=0.004082 source_path='example_assets/08_end_to_end_omop/patients.csv' source_rows_read=15 output_path='./omop_output' tables=[TableResult(table_name='person', rows_written=15, columns=['person_id', 'person_source_value', 'birth_datetime', 'gender_concept_id', 'race_concept_id', 'ethnicity_concept_id'], output_path='./omop_output/person.csv', concept_columns_mapped=[], unmapped_concept_count=0), TableResult(table_name='location', rows_written=15, columns=['address_1', 'city', 'state', 'zip'], output_path='./omop_output/location.csv', concept_columns_mapped=[], unmapped_concept_count=0)] total_rows_written=30 schema_mappings_applied=10 concept_mappings_applied=0 unmapped_columns=['last_name'] engine_name='polars' errors

In [15]:
# Run ETL for encounters
encounters_etl = project.run_etl(
    encounters,
    output_dir="./omop_output",
    schema_mapping=encounters_schema,
    concept_mapping=dx_concept_map,
)

print("Encounters ETL completed.")

2026-04-16 00:28:09 [info     ] Reading source                 format=csv path=example_assets/08_end_to_end_omop/encounters.csv


2026-04-16 00:28:09 [info     ] Processing table               columns=6 table=visit_occurrence


2026-04-16 00:28:09 [info     ] Processing table               columns=1 table=person


2026-04-16 00:28:09 [info     ] ETL complete                   duration=0.002477 success=True


2026-04-16 00:28:09 [info     ] project.etl_complete           output_dir=./omop_output project='OMOP End-to-End Demo'


Encounters ETL completed.


In [16]:
# Run ETL for diagnoses
diagnoses_etl = project.run_etl(
    diagnoses,
    output_dir="./omop_output",
    schema_mapping=diagnoses_schema,
    concept_mapping=dx_concept_map,
)

print("Diagnoses ETL completed.")

2026-04-16 00:28:09 [info     ] Reading source                 format=csv path=example_assets/08_end_to_end_omop/diagnoses.csv


2026-04-16 00:28:09 [info     ] Processing table               columns=1 table=visit_occurrence


2026-04-16 00:28:09 [info     ] Processing table               columns=1 table=person


2026-04-16 00:28:09 [info     ] Processing table               columns=3 table=condition_occurrence


2026-04-16 00:28:09 [info     ] ETL complete                   duration=0.003048 success=True


2026-04-16 00:28:09 [info     ] project.etl_complete           output_dir=./omop_output project='OMOP End-to-End Demo'


Diagnoses ETL completed.


In [17]:
# Run ETL for medications
medications_etl = project.run_etl(
    medications,
    output_dir="./omop_output",
    schema_mapping=medications_schema,
    concept_mapping=rx_concept_map,
)

print("Medications ETL completed.")

2026-04-16 00:28:09 [info     ] Reading source                 format=csv path=example_assets/08_end_to_end_omop/medications.csv


2026-04-16 00:28:09 [info     ] Processing table               columns=5 table=drug_exposure


2026-04-16 00:28:09 [info     ] Processing table               columns=1 table=person


2026-04-16 00:28:09 [info     ] ETL complete                   duration=0.002414 success=True


2026-04-16 00:28:09 [info     ] project.etl_complete           output_dir=./omop_output project='OMOP End-to-End Demo'


Medications ETL completed.


## Stage 6: Validate Output

Validation checks the generated OMOP tables for:
- Required columns are present
- Data types match the CDM specification
- Foreign key relationships are valid
- Concept IDs reference valid standard concepts
- Date ranges are plausible
- No orphaned records

In [18]:
validation = project.validate(etl_result=etl_result)

print("Validation Results")
print("=" * 50)
print(f"All tables passed: {validation['all_passed']}")
print()
for table in validation['tables']:
    status = "PASS" if table.get('passed') else "FAIL"
    table_name = table.get('table_name', 'unknown')
    print(f"  {table_name:30s} {status}")
    if not table.get('passed') and table.get('errors'):
        for error in table['errors']:
            print(f"    - {error}")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

2026-04-16 00:28:09 [info     ] gx_validator.validated         completeness=0.45 conformance=1.00 passed=False plausibility=0.50 table=drug_exposure


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:28:09 [info     ] gx_validator.validated         completeness=0.44 conformance=1.00 passed=False plausibility=0.44 table=location


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

2026-04-16 00:28:09 [info     ] gx_validator.validated         completeness=0.11 conformance=1.00 passed=False plausibility=0.11 table=visit_occurrence


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:28:09 [info     ] gx_validator.validated         completeness=0.11 conformance=1.00 passed=False plausibility=0.11 table=person


2026-04-16 00:28:09 [info     ] project.validated              all_passed=False project='OMOP End-to-End Demo' tables=5


Validation Results
All tables passed: False

  drug_exposure                  FAIL
  location                       FAIL
  condition_occurrence           FAIL
  visit_occurrence               FAIL
  person                         FAIL


In [19]:
# Validate the other ETL results as well
for name, result in [("encounters", encounters_etl), ("diagnoses", diagnoses_etl), ("medications", medications_etl)]:
    v = project.validate(etl_result=result)
    status = "ALL PASSED" if v['all_passed'] else "SOME FAILED"
    print(f"{name:15s}: {status}")
    for table in v['tables']:
        if not table.get('passed'):
            print(f"  FAIL: {table.get('table_name', 'unknown')}")
            for error in table.get('errors', []):
                print(f"    - {error}")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

2026-04-16 00:28:09 [info     ] gx_validator.validated         completeness=0.45 conformance=1.00 passed=False plausibility=0.50 table=drug_exposure


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:28:10 [info     ] gx_validator.validated         completeness=0.44 conformance=1.00 passed=False plausibility=0.44 table=location


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...

2026-04-16 00:28:11 [info     ] gx_validator.validated         completeness=0.11 conformance=1.00 passed=False plausibility=0.11 table=visit_occurrence


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:28:11 [info     ] gx_validator.validated         completeness=0.11 conformance=1.00 passed=False plausibility=0.11 table=person


2026-04-16 00:28:11 [info     ] project.validated              all_passed=False project='OMOP End-to-End Demo' tables=5


medications    : SOME FAILED
  FAIL: drug_exposure
  FAIL: location
  FAIL: condition_occurrence
  FAIL: visit_occurrence
  FAIL: person


## Stage 7: Inspect Output Files

Let us examine the generated OMOP CDM tables to verify the transformation.

In [20]:
import pandas as pd
import os

output_dir = "./omop_output"

# List all generated files
if os.path.exists(output_dir):
    files = sorted(os.listdir(output_dir))
    print(f"Generated OMOP CDM files in {output_dir}:")
    print("-" * 50)
    for f in files:
        filepath = os.path.join(output_dir, f)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  {f:35s} {size_kb:8.1f} KB")
else:
    print(f"Output directory {output_dir} does not exist yet.")
    print("Run the ETL cells above first.")

Generated OMOP CDM files in ./omop_output:
--------------------------------------------------
  condition_occurrence.csv                 0.6 KB
  drug_exposure.csv                        0.6 KB
  location.csv                             0.5 KB
  person.csv                               0.1 KB
  visit_occurrence.csv                     0.1 KB


In [21]:
# Preview the person table
person_file = os.path.join(output_dir, "person.csv")
if os.path.exists(person_file):
    person_df = pd.read_csv(person_file)
    print(f"person table: {len(person_df)} rows, {len(person_df.columns)} columns")
    print(f"Columns: {list(person_df.columns)}")
    print()
    print(person_df.head().to_string(index=False))
else:
    print("person.csv not found. Run the ETL cells above first.")

person table: 15 rows, 1 columns
Columns: ['person_id']

person_id
     P001
     P001
     P002
     P003
     P003


In [22]:
# Preview the condition_occurrence table
condition_file = os.path.join(output_dir, "condition_occurrence.csv")
if os.path.exists(condition_file):
    condition_df = pd.read_csv(condition_file)
    print(f"condition_occurrence table: {len(condition_df)} rows, {len(condition_df.columns)} columns")
    print(f"Columns: {list(condition_df.columns)}")
    print()
    print(condition_df.head().to_string(index=False))
else:
    print("condition_occurrence.csv not found. Run the ETL cells above first.")

condition_occurrence table: 20 rows, 3 columns
Columns: ['condition_source_value', 'condition_type_concept_id', 'condition_start_date']

condition_source_value condition_type_concept_id condition_start_date
                 E11.9                   primary           2024-01-15
                   I10                 secondary           2024-01-15
                 J06.9                   primary           2024-02-03
                 J44.1                   primary           2024-01-22
                 N18.3                 secondary           2024-01-22


In [23]:
# Preview the drug_exposure table
drug_file = os.path.join(output_dir, "drug_exposure.csv")
if os.path.exists(drug_file):
    drug_df = pd.read_csv(drug_file)
    print(f"drug_exposure table: {len(drug_df)} rows, {len(drug_df.columns)} columns")
    print(f"Columns: {list(drug_df.columns)}")
    print()
    print(drug_df.head().to_string(index=False))
else:
    print("drug_exposure.csv not found. Run the ETL cells above first.")

drug_exposure table: 15 rows, 5 columns
Columns: ['drug_exposure_id', 'drug_source_value', 'quantity', 'drug_exposure_start_date', 'provider_id']

drug_exposure_id drug_source_value  quantity drug_exposure_start_date provider_id
           RX001            MET500        60               2024-01-15       DR001
           RX002           LISIN10        30               2024-01-15       DR001
           RX003           AMOX500        21               2024-02-03       DR002
           RX004             AMLO5        30               2024-01-22       DR003
           RX005            ATOR20        30               2024-01-22       DR003


In [24]:
# Export the source_to_concept_map for both domains
dx_stcm = dx_concept_map.to_source_to_concept_map()
rx_stcm = rx_concept_map.to_source_to_concept_map()

all_stcm = dx_stcm + rx_stcm
stcm_df = pd.DataFrame(all_stcm)

print(f"Combined source_to_concept_map: {len(stcm_df)} entries")
print()
print(stcm_df.to_string(index=False))

# Save to output directory
stcm_path = os.path.join(output_dir, "source_to_concept_map.csv")
stcm_df.to_csv(stcm_path, index=False)
print(f"\nSaved to {stcm_path}")

Combined source_to_concept_map: 15 entries

source_code  source_concept_id source_vocabulary_id                                            source_description                                       source_code_description  target_concept_id                                         target_concept_name target_vocabulary_id  confidence method valid_start_date valid_end_date invalid_reason
      E11.9                  0             Hospital               Type 2 diabetes mellitus, without complications               Type 2 diabetes mellitus, without complications           44787902         Lipoatrophic diabetes mellitus without complication               SNOMED   10.359390   auto       1970-01-01     2099-12-31           None
        I10                  0             Hospital                              Essential (primary) hypertension                              Essential (primary) hypertension             320128                                      Essential hypertension               SNO

## Summary

This notebook demonstrated the complete Portiere pipeline for transforming hospital
data into the OMOP CDM v5.4. The key stages were:

1. **Configuration** -- Set up custom thresholds for schema and concept mapping
2. **Ingestion** -- Loaded four source files (patients, encounters, diagnoses, medications)
3. **Schema Mapping** -- Mapped source columns to OMOP CDM table columns
4. **Concept Mapping** -- Translated ICD-10 codes to SNOMED and local drug codes to RxNorm
5. **ETL Execution** -- Generated OMOP-compliant output files
6. **Validation** -- Verified output tables against CDM specifications

### OMOP CDM Table Reference

The tables generated in this pipeline correspond to the OMOP CDM v5.4 clinical data tables:

| OMOP Table | Description | Primary Key |
|---|---|---|
| `person` | Patient demographics (birth date, gender, race, ethnicity) | `person_id` |
| `visit_occurrence` | Hospital encounters (admissions, outpatient visits, ER) | `visit_occurrence_id` |
| `condition_occurrence` | Diagnoses and conditions recorded during visits | `condition_occurrence_id` |
| `drug_exposure` | Medication orders, prescriptions, and administrations | `drug_exposure_id` |
| `measurement` | Lab results, vitals, and other quantitative observations | `measurement_id` |
| `observation` | Non-standard clinical observations | `observation_id` |
| `procedure_occurrence` | Surgical and diagnostic procedures | `procedure_occurrence_id` |
| `source_to_concept_map` | Documents how source codes were mapped to standard concepts | composite |

### Next Steps

After completing the ETL transformation, recommended next steps include:

1. **Load into a CDM database** -- Import the CSV files into a PostgreSQL or SQL Server database
   with the OMOP CDM schema.

2. **Run ACHILLES** -- Execute the OHDSI ACHILLES tool to generate data characterization reports
   and identify data quality issues.

3. **Data Quality Dashboard** -- Use the OHDSI Data Quality Dashboard to run a comprehensive
   suite of data quality checks against the CDM.

4. **ATLAS** -- Connect to ATLAS for cohort definition, characterization, and
   population-level estimation studies.

5. **Iterate on mappings** -- Refine concept mappings based on data quality findings.
   Pay special attention to high-frequency unmapped codes and domain mismatches.